In [17]:
import os
import json
import joblib
import pandas as pd
import langchain
import langchain_community
import langchain_google_genai
import chromadb
import pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate

DATASET_DIR = r"G:\Projects\Depi\Depi Grid\Datasets\unisolar"
MODEL_PATH = r"G:\Projects\Depi\Depi Grid\models\Solar Forecast\solar_theoretical_yield_model (Anomaly).pkl"
MANUAL_PATH = os.path.join(DATASET_DIR, r"G:\Projects\Depi\Depi Grid\knowledge_base\manuals\ABB_PVS800_Manual.pdf")

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6KPbsPaW6fZyQp8N5jvdG_HM5cg_xnnmVH-XZBhQmKgSg"

In [ ]:
print("Ingesting the ABB PVS800 Inverter Manual")
loader = PyPDFLoader(MANUAL_PATH)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splits = text_splitter.split_documents(pages)
print(f"Split manual into {len(splits)} chunks.")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

import time
BATCH_SIZE = 90
SLEEP_SECONDS = 65  

vectorstore = None
for i in range(0, len(splits), BATCH_SIZE):
    batch = splits[i:i + BATCH_SIZE]
    print(f"Embedding chunks {i}-{i+len(batch)-1} of {len(splits)}")

    if vectorstore is None:
        vectorstore = Chroma.from_documents(documents=batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)

    if i + BATCH_SIZE < len(splits):
        print(f"Pausing {SLEEP_SECONDS}s to stay under the free tier rate limit")
        time.sleep(SLEEP_SECONDS)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Vector database built successfully with {len(splits)} document chunks.")


Ingesting the ABB PVS800 Inverter Manual
Split manual into 242 chunks.
Embedding chunks 0-89 of 242...
Pausing 65s to stay under the free-tier rate limit...
Embedding chunks 90-179 of 242...
Pausing 65s to stay under the free-tier rate limit...
Embedding chunks 180-241 of 242...
Vector database built successfully with 242 document chunks.


In [19]:
class RAGDiagnosticEngine:
    def __init__(self, baseline_model_path, document_retriever):
        self.yield_model = joblib.load(baseline_model_path)
        self.retriever = document_retriever
        
        self.llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.0)
        
        self.prompt = PromptTemplate.from_template(
            "You are an expert solar grid diagnostic AI. Analyze the telemetry data and use ONLY the provided Inverter Manual excerpts to diagnose the issue.\n\n"
            "INVERTER MANUAL EXCERPTS:\n{context}\n\n"
            "LIVE TELEMETRY DATA:\n{telemetry}\n\n"
            "Respond with a raw JSON object containing exactly these keys: STATUS (HEALTHY, WARNING_MAINTENANCE, or CRITICAL_FAULT), ERROR_CODE, METRIC_EXPLANATION, RECOMMENDED_ACTION. Do not include markdown formatting like ```json."
        )

    def diagnose_interval(self, row):
        X = pd.DataFrame([{
            'IRRADIATION': row['IRRADIATION'],
            'MODULE_TEMPERATURE': row['MODULE_TEMPERATURE'],
            'AMBIENT_TEMPERATURE': row['AMBIENT_TEMPERATURE']
        }])
        
        expected_dc = self.yield_model.predict(X)[0]
        actual_dc = row['DC_POWER']
        power_deficit = expected_dc - actual_dc
        efficiency_loss = power_deficit / (expected_dc + 1e-5)

        if efficiency_loss < 0.10:
            return {
                "STATUS": "HEALTHY",
                "ERROR_CODE": "ERR_0000_NONE",
                "METRIC_EXPLANATION": f"Operating normally. Expected: {expected_dc:.2f}kW, Actual: {actual_dc:.2f}kW.",
                "RECOMMENDED_ACTION": "No action required."
            }

        telemetry_query = (
            f"Inverter ID: {row.get('SOURCE_KEY', 'UNKNOWN')}\n"
            f"Irradiation: {row['IRRADIATION']} W/m2\n"
            f"Module Temp: {row['MODULE_TEMPERATURE']} C\n"
            f"Ambient Temp: {row['AMBIENT_TEMPERATURE']} C\n"
            f"Expected Power: {expected_dc:.2f} kW\n"
            f"Actual Power: {actual_dc:.2f} kW\n"
            f"Efficiency Loss: {efficiency_loss*100:.1f}%\n"
        )

        docs = self.retriever.invoke(telemetry_query)
        context = "\n\n".join([d.page_content for d in docs])


        chain = self.prompt | self.llm
        response = chain.invoke({"context": context, "telemetry": telemetry_query})
        
        raw_content = response.content
        if isinstance(raw_content, list):
            text_parts = []
            for part in raw_content:
                if isinstance(part, str):
                    text_parts.append(part)
                elif isinstance(part, dict):
                    text_parts.append(part.get("text", ""))
            raw_content = "".join(text_parts)

        cleaned = raw_content.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            if cleaned.startswith("json"):
                cleaned = cleaned[4:].strip()

        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return {
                "STATUS": "PARSE_ERROR", 
                "METRIC_EXPLANATION": "Failed to parse LLM response.", 
                "RAW_OUTPUT": raw_content
            }

In [ ]:
df_raw = pd.read_csv(os.path.join(DATASET_DIR, "Plant_1_Generation_Data.csv"))
df_weather = pd.read_csv(os.path.join(DATASET_DIR, "Plant_1_Weather_Sensor_Data.csv"))

df_raw['DATE_TIME'] = pd.to_datetime(df_raw['DATE_TIME'], format='%d-%m-%Y %H:%M')
df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
df_merged = pd.merge(df_raw, df_weather, on='DATE_TIME', how='inner')


engine = RAGDiagnosticEngine(MODEL_PATH, retriever)

healthy_production_df = df_merged[(df_merged['DC_POWER'] > 5000) & (df_merged['AC_POWER'] > 400)]
sample_baseline_row = healthy_production_df.iloc[0].copy()


sample_fault_row = sample_baseline_row.copy()
sample_fault_row['MODULE_TEMPERATURE'] = 65.5  #overheating 
sample_fault_row['DC_POWER'] = sample_fault_row['DC_POWER'] * 0.4  #efficiency drop

print("Routing anomalous telemetry to RAG Engine for manual-based diagnosis\n")
explanation = engine.diagnose_interval(sample_fault_row)

print(json.dumps(explanation, indent=4))

Routing anomalous telemetry to RAG Engine for manual-based diagnosis...

{
    "STATUS": "CRITICAL_FAULT",
    "ERROR_CODE": "ERR_EFFICIENCY_DEGRADATION",
    "METRIC_EXPLANATION": "The telemetry data indicates an actual power output of 2029.20 kW against an expected 4654.48 kW, resulting in a critical efficiency loss of 56.4%. According to the PVS800 manual (pages 143-145), all PVS800-57 inverter models are rated for a European efficiency between 95.3% and 98.6%, and a maximum efficiency between 96.6% and 98.8% across their respective DC voltage ranges (450V to 850V). A 56.4% efficiency loss is a severe deviation from these technical specifications. Furthermore, the module temperature is highly elevated at 65.5 C compared to the ambient temperature of 25.96 C.",
    "RECOMMENDED_ACTION": "Initiate an immediate emergency inspection of the inverter units. Check the cooling and ventilation systems to address the high module temperature of 65.5 C, which may be causing thermal derating or 